# Stage 02 · Step 3 — Surrogate-driven τ search

Use the SSL+RUL model to predict component-level remaining life on observed telemetry, derive an expected number of preventive / corrective events for any candidate τ schedule, and let Optuna optimise τ over that surrogate without paying the simulator's full cost.

The surrogate's predictions are validated against the **real simulator** at the end: the top-K τ vectors found by the surrogate are re-simulated with `lib.env_runner.run_with_tau` and scored with `lib.objective.scalar_objective`, then compared against Stage 01's leaderboard.

If the surrogate is well-calibrated, Stage 02 returns a τ vector whose true cost is within a few percent of Stage 01's optimum — at a fraction of the wall-clock time spent inside the optimiser.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from transformers import PatchTSTConfig, PatchTSTForRegression

from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TEST_PRINTERS,
    TRAIN_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.env_runner import default_dates, run_with_tau
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.objective import scalar_objective
from ml_models import PROJECT_ROOT
from sdg.generate import build_printer_city_map, load_configs
from sdg.schema import COMPONENT_IDS

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
RESULTS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from ml_models.lib.fast import SURROGATE_OPTUNA_TRIALS, PARALLEL, banner
banner()


[fast] mode=FULL · parallel=12 · trials=200/500 · K=60 · epochs=20/3 · ppo_ts=2000/20000 · seeds=(0, 1, 2)


In [2]:
with (MODELS_DIR / 'ssl_config.json').open() as handle:
    saved = json.load(handle)
patch_cfg = PatchTSTConfig(**saved['patch_cfg'])
patch_cfg.num_targets = 6
patch_cfg.prediction_length = 1
patch_cfg.use_cls_token = False
model = PatchTSTForRegression(patch_cfg).to(device)
model.load_state_dict(torch.load(MODELS_DIR / 'rul_head_ssl.pt', map_location=device))
model.eval()

scaler = np.load(MODELS_DIR / 'feature_scaler.npz', allow_pickle=True)
channel_mean = scaler['mean']; channel_std = scaler['std']
CONTEXT_LENGTH = int(saved['train_cfg']['context_length'])
print('Loaded RUL model. Context length:', CONTEXT_LENGTH)

[transformers] Setting `do_mask_input` parameter to False.


Loaded RUL model. Context length: 360


In [3]:
fleet = load_fleet(DEFAULT_FLEET_PATH)
feat_fleet, feature_cols = build_feature_matrix(fleet)

def predict_rul_for_window(printer_id: int, end_day: int) -> np.ndarray:
    df = filter_printers(feat_fleet, [printer_id])
    df = df.sort_values('day')
    arr = df[feature_cols].to_numpy(dtype=np.float32)
    if end_day < CONTEXT_LENGTH:
        return np.full(6, np.nan, dtype=np.float32)
    window = arr[end_day - CONTEXT_LENGTH:end_day]
    normed = (window - channel_mean) / channel_std
    x = torch.from_numpy(normed).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(past_values=x).regression_outputs.squeeze(-1).view(-1)
    return (out.cpu().numpy() * 365.0).clip(0.0, 365.0)

sample_rul = predict_rul_for_window(VAL_PRINTERS[0], end_day=2000)
print('Predicted RUL for printer', VAL_PRINTERS[0], 'at day 2000:', sample_rul)

Predicted RUL for printer 70 at day 2000: [  0.         23.296179    0.          7.8708444 100.26616   365.       ]


## Surrogate cost model

For each candidate τ vector and each held-out printer, estimate annual preventive and corrective events using:

$$n_{prev,i} \approx H \cdot (24 / \tau_i) \qquad n_{corr,i} \approx H \cdot \hat{\lambda}_{cm,i}(\tau_i)$$

where $H$ is the simulated horizon (years), and $\hat{\lambda}_{cm,i}$ is a smooth function of $\tau_i$ fit from the RUL predictions: rapidly decreasing for short τ (frequent maintenance prevents failures), increasing for long τ (more failures slip through). The simplest such fit uses a single observation: at the baseline τ, observed corrective events are known from `fleet_baseline.parquet`; at τ→∞, every component eventually fails and contributes a corrective event per nominal life. We interpolate between these regimes via the predicted RUL on validation printers.

In [4]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
components = components_cfg['components']
DATES = default_dates()
HORIZON_YEARS = len(DATES) / 365.25

anchor_pm: dict[str, int] = {}
anchor_cm: dict[str, int] = {}
for component_id in COMPONENT_IDS:
    anchor_pm[component_id] = int(filter_printers(fleet, list(TRAIN_PRINTERS) + list(VAL_PRINTERS))[f'maint_{component_id}'].sum())
    anchor_cm[component_id] = int(filter_printers(fleet, list(TRAIN_PRINTERS) + list(VAL_PRINTERS))[f'failure_{component_id}'].sum())
anchor_tau = {component_id: float(components[component_id]['tau_nom_d']) for component_id in COMPONENT_IDS}
anchor_pm, anchor_cm, anchor_tau

({'C1': 61, 'C2': 504, 'C3': 1257, 'C4': 545, 'C5': 295, 'C6': 761},
 {'C1': 165682, 'C2': 1866, 'C3': 226983, 'C4': 23042, 'C5': 2667, 'C6': 174},
 {'C1': 25.0,
  'C2': 166.6666667,
  'C3': 7.0,
  'C4': 41.6666667,
  'C5': 166.6666667,
  'C6': 333.3333333})

In [5]:
def surrogate_score(tau_vector: dict[str, float]) -> dict:
    # event counts pool TRAIN+VAL, so normalise to the same denominator
    n_printers = len(TRAIN_PRINTERS) + len(VAL_PRINTERS)
    pm_total = 0.0
    cm_total = 0.0
    downtime_total = 0.0
    for component_id in COMPONENT_IDS:
        spec = components[component_id]
        tau_new = float(tau_vector[component_id])
        tau_base = anchor_tau[component_id]
        ratio = tau_base / tau_new if tau_new > 0 else 1.0
        # Linear scaling of preventive events with maintenance frequency.
        pm_count = anchor_pm[component_id] * ratio
        # Failure rate inflates with τ: when maintenance is rarer, failures grow.
        cm_count = anchor_cm[component_id] * (tau_new / tau_base) ** float(spec.get('b_M', 1.5))
        # Add an upper bound: at most one corrective per nominal life per printer.
        cm_cap = n_printers * (HORIZON_YEARS * 365.25 / float(spec['L_nom_d']))
        cm_count = min(cm_count, cm_cap)
        pm_total += pm_count * float(spec['cost_preventive_eur'])
        cm_total += cm_count * float(spec['cost_corrective_eur'])
        downtime_total += pm_count * float(spec['downtime_preventive_d']) * 24
        downtime_total += cm_count * float(spec['downtime_corrective_d']) * 24
    norm = n_printers * HORIZON_YEARS
    annual_cost = (pm_total + cm_total) / norm
    total_hours = len(DATES) * 24.0 * n_printers
    availability = (total_hours - downtime_total) / total_hours if total_hours > 0 else 1.0
    deficit = max(0.0, 0.95 - availability)
    return {
        'value': annual_cost + 1_000_000.0 * deficit,
        'annual_cost': annual_cost,
        'availability': availability,
    }

surrogate_score({c: anchor_tau[c] for c in COMPONENT_IDS})

{'value': 165265.275265405,
 'annual_cost': 165265.275265405,
 'availability': 0.954063482613818}

In [6]:
TAU_RANGES = {
    'C1': (2.08, 83.33),
    'C2': (20.83, 833.33),
    'C3': (1.0, 20.83),
    'C4': (4.17, 83.33),
    'C5': (20.83, 333.33),
    'C6': (41.67, 833.33),
}

def trial_to_tau(trial: optuna.Trial) -> dict[str, float]:
    return {c: trial.suggest_float(f'tau_{c}', low, high, log=True) for c, (low, high) in TAU_RANGES.items()}

def surrogate_objective(trial: optuna.Trial) -> float:
    tau_vector = trial_to_tau(trial)
    score = surrogate_score(tau_vector)
    for key in ('annual_cost', 'availability'):
        trial.set_user_attr(key, float(score[key]))
    return float(score['value'])

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=99),
)
study.optimize(surrogate_objective, n_trials=SURROGATE_OPTUNA_TRIALS, n_jobs=PARALLEL, show_progress_bar=True)  # was n_trials=500, n_jobs=1
study_df = study.trials_dataframe().sort_values('value').reset_index(drop=True)
study_df.head(5)

[I 2026-04-25 22:36:58,679] A new study created in memory with name: no-name-1c496966-b075-4b58-9e36-1682e4263e32


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-04-25 22:36:58,682] Trial 0 finished with value: 160978.02002533578 and parameters: {'tau_C1': 12.8506629598033, 'tau_C2': 65.17709153813125, 'tau_C3': 12.61791583937171, 'tau_C4': 42.16718152628861, 'tau_C5': 22.7912709611736, 'tau_C6': 330.3493878315024}. Best is trial 0 with value: 160978.02002533578.
[I 2026-04-25 22:36:58,683] Trial 1 finished with value: 173138.49442182577 and parameters: {'tau_C1': 11.852054162411052, 'tau_C2': 815.8575961416498, 'tau_C3': 2.878021756135309, 'tau_C4': 30.027424013481212, 'tau_C5': 246.79756535307268, 'tau_C6': 105.76240740907903}. Best is trial 0 with value: 160978.02002533578.
[I 2026-04-25 22:36:58,683] Trial 2 finished with value: 159826.14930097296 and parameters: {'tau_C1': 4.6439488046885975, 'tau_C2': 31.927940451170166, 'tau_C3': 8.866197678258658, 'tau_C4': 65.97378250814164, 'tau_C5': 30.09650236991545, 'tau_C6': 294.3066430781623}. Best is trial 2 with value: 159826.14930097296.
[I 2026-04-25 22:36:58,684] Trial 3 finished wit

[I 2026-04-25 22:36:58,684] Trial 4 finished with value: 164791.449122134 and parameters: {'tau_C1': 40.38217449715066, 'tau_C2': 665.3410570148482, 'tau_C3': 13.687850658754748, 'tau_C4': 21.63940660207554, 'tau_C5': 166.65128327691508, 'tau_C6': 325.213263159727}. Best is trial 2 with value: 159826.14930097296.
[I 2026-04-25 22:36:58,685] Trial 5 finished with value: 177758.2981637706 and parameters: {'tau_C1': 10.68320949908082, 'tau_C2': 500.3820364751523, 'tau_C3': 8.383684389225076, 'tau_C4': 8.067641170950326, 'tau_C5': 41.881988716546736, 'tau_C6': 44.38528942686943}. Best is trial 2 with value: 159826.14930097296.
[I 2026-04-25 22:36:58,685] Trial 6 finished with value: 171201.40569655452 and parameters: {'tau_C1': 6.6428842261558705, 'tau_C2': 522.7266050161728, 'tau_C3': 17.31974018720762, 'tau_C4': 27.74205805221559, 'tau_C5': 86.6582332575298, 'tau_C6': 47.69500660365176}. Best is trial 2 with value: 159826.14930097296.
[I 2026-04-25 22:36:58,686] Trial 7 finished with val

[I 2026-04-25 22:36:58,887] Trial 50 finished with value: 166134.04946320064 and parameters: {'tau_C1': 5.473255635507145, 'tau_C2': 82.50644948964923, 'tau_C3': 4.1476935138713715, 'tau_C4': 34.090282400465846, 'tau_C5': 26.749732709923283, 'tau_C6': 349.65563332105387}. Best is trial 21 with value: 138803.64382930985.
[I 2026-04-25 22:36:58,887] Trial 51 finished with value: 168302.36342852062 and parameters: {'tau_C1': 5.029761895812011, 'tau_C2': 90.84740175210275, 'tau_C3': 4.060623885368108, 'tau_C4': 31.833711429793055, 'tau_C5': 110.66092529081604, 'tau_C6': 572.5406630780295}. Best is trial 21 with value: 138803.64382930985.


...

[I 2026-04-25 22:37:01,180] Trial 407 finished with value: 136952.58915855733 and parameters: {'tau_C1': 2.2235570464051064, 'tau_C2': 29.317012430442645, 'tau_C3': 18.537001223734354, 'tau_C4': 81.25335954160128, 'tau_C5': 23.45107329425052, 'tau_C6': 620.1913798514163}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,195] Trial 409 finished with value: 138971.23466311864 and parameters: {'tau_C1': 2.2062307124432112, 'tau_C2': 29.630956470552007, 'tau_C3': 18.821397916342455, 'tau_C4': 79.68718114821775, 'tau_C5': 245.06394224698138, 'tau_C6': 614.7518906030188}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,204] Trial 411 finished with value: 135370.85609488923 and parameters: {'tau_C1': 2.0920616904296048, 'tau_C2': 30.307189740118762, 'tau_C3': 18.62956832743798, 'tau_C4': 83.13589675431717, 'tau_C5': 23.621364911886328, 'tau_C6': 615.5206852700069}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,205] Tr

[I 2026-04-25 22:37:01,395] Trial 434 finished with value: 143073.21241728665 and parameters: {'tau_C1': 2.530060025919976, 'tau_C2': 33.58081725975293, 'tau_C3': 20.774889362631313, 'tau_C4': 75.96380559489671, 'tau_C5': 127.91779720805602, 'tau_C6': 724.3965891270053}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,403] Trial 436 finished with value: 142126.43637835816 and parameters: {'tau_C1': 2.6368279563858255, 'tau_C2': 34.04238956450865, 'tau_C3': 19.671406519911823, 'tau_C4': 82.88052816436851, 'tau_C5': 27.094476384294563, 'tau_C6': 717.1716715363522}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,408] Trial 437 finished with value: 157128.75305191654 and parameters: {'tau_C1': 12.819279250427305, 'tau_C2': 25.544202907036226, 'tau_C3': 19.81227582057586, 'tau_C4': 74.9698374843144, 'tau_C5': 26.626113494963732, 'tau_C6': 589.3471949858784}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,412] Trial

[I 2026-04-25 22:37:01,600] Trial 459 finished with value: 139160.99192160944 and parameters: {'tau_C1': 2.392213158961176, 'tau_C2': 22.681911108545744, 'tau_C3': 18.577245980688193, 'tau_C4': 71.3911826157034, 'tau_C5': 23.376332259392687, 'tau_C6': 694.3093892331229}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,606] Trial 460 finished with value: 141085.4685135967 and parameters: {'tau_C1': 2.3902698197936805, 'tau_C2': 152.5428057835919, 'tau_C3': 20.798829986611977, 'tau_C4': 79.68884541670606, 'tau_C5': 23.95047205077131, 'tau_C6': 642.5750499123081}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,629] Trial 461 finished with value: 139240.153419848 and parameters: {'tau_C1': 2.40047851890264, 'tau_C2': 22.76726252125466, 'tau_C3': 18.524807169334608, 'tau_C4': 79.7925296214052, 'tau_C5': 23.84598399965793, 'tau_C6': 625.2474356709511}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,629] Trial 464 fi

[I 2026-04-25 22:37:01,821] Trial 487 finished with value: 148822.4821860627 and parameters: {'tau_C1': 2.085020795232666, 'tau_C2': 26.498164434277395, 'tau_C3': 1.9032466852952077, 'tau_C4': 30.68155481020698, 'tau_C5': 27.528124153213753, 'tau_C6': 741.1413488772519}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,823] Trial 490 finished with value: 157041.61418805728 and parameters: {'tau_C1': 72.169743318636, 'tau_C2': 26.884630135852138, 'tau_C3': 17.544523137218224, 'tau_C4': 83.29377166719408, 'tau_C5': 27.77739510167605, 'tau_C6': 745.4504438102226}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,829] Trial 489 finished with value: 135431.90154326978 and parameters: {'tau_C1': 2.0843642004701426, 'tau_C2': 26.549303077660912, 'tau_C3': 17.58548342516196, 'tau_C4': 74.6834200092106, 'tau_C5': 28.124470356001467, 'tau_C6': 737.6459838796517}. Best is trial 403 with value: 135043.85112705317.
[I 2026-04-25 22:37:01,830] Trial 48

,number,value,datetime_start,datetime_complete,duration,params_tau_C1,params_tau_C2,params_tau_C3,params_tau_C4,params_tau_C5,params_tau_C6,user_attrs_annual_cost,user_attrs_availability,state
0,403,135043.851127,2026-04-25 22:37:01.058065,2026-04-25 22:37:01.146337,0 days 00:00:00.088272,2.080770,30.108913,19.243376,82.963079,23.522210,775.703691,135043.851127,0.964548,COMPLETE
1,442,135112.348842,2026-04-25 22:37:01.364951,2026-04-25 22:37:01.456589,0 days 00:00:00.091638,2.081791,24.882185,19.680343,82.282606,22.443172,714.215837,135112.348842,0.964394,COMPLETE
2,371,135173.895363,2026-04-25 22:37:00.807784,2026-04-25 22:37:00.890362,0 days 00:00:00.082578,2.080086,27.422296,20.811095,77.201898,28.182554,677.862020,135173.895363,0.964470,COMPLETE
3,482,135184.960188,2026-04-25 22:37:01.692532,2026-04-25 22:37:01.775097,0 days 00:00:00.082565,2.083276,25.900568,19.678728,76.270575,26.297860,748.284277,135184.960188,0.964448,COMPLETE
4,418,135186.934920,2026-04-25 22:37:01.173207,2026-04-25 22:37:01.259885,0 days 00:00:00.086678,2.084415,31.507133,18.541069,79.518243,23.478456,775.768345,135186.934920,0.964538,COMPLETE


In [7]:
TOP_K = 5
real_results = []
for _, row in study_df.head(TOP_K).iterrows():
    tau_vector = {c: float(row[f'params_tau_{c}']) for c in COMPONENT_IDS}
    events = run_with_tau(
        tau_vector,
        printer_ids=TEST_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    real_score = scalar_objective(events, components_cfg)
    real_results.append({
        'trial': int(row['number']),
        'surrogate_value': float(row['value']),
        'surrogate_annual_cost': float(row['user_attrs_annual_cost']),
        'real_value': float(real_score['value']),
        'real_annual_cost': float(real_score['annual_cost']),
        'real_availability': float(real_score['availability']),
        **{f'tau_{c}': tau_vector[c] for c in COMPONENT_IDS},
    })
real_df = pd.DataFrame(real_results).sort_values('real_value').reset_index(drop=True)
real_df

,trial,surrogate_value,surrogate_annual_cost,real_value,real_annual_cost,real_availability,tau_C1,tau_C2,tau_C3,tau_C4,tau_C5,tau_C6
0,403,135043.851127,135043.851127,1.050000e+10,4.044317e+06,0.0,2.080770,30.108913,19.243376,82.963079,23.522210,775.703691
1,442,135112.348842,135112.348842,1.050000e+10,4.043827e+06,0.0,2.081791,24.882185,19.680343,82.282606,22.443172,714.215837
2,371,135173.895363,135173.895363,1.050000e+10,4.032210e+06,0.0,2.080086,27.422296,20.811095,77.201898,28.182554,677.862020
3,482,135184.960188,135184.960188,1.050000e+10,4.041440e+06,0.0,2.083276,25.900568,19.678728,76.270575,26.297860,748.284277
4,418,135186.934920,135186.934920,1.050000e+10,4.050124e+06,0.0,2.084415,31.507133,18.541069,79.518243,23.478456,775.768345


In [8]:
winner = real_df.iloc[0]
best_tau = {c: float(winner[f'tau_{c}']) for c in COMPONENT_IDS}
payload = {
    'tau_nom_h': best_tau,
    'validated_on': 'test printers (id 85..99)',
    'surrogate_annual_cost': float(winner['surrogate_annual_cost']),
    'real_annual_cost_eur_per_printer_year': float(winner['real_annual_cost']),
    'real_availability': float(winner['real_availability']),
}
with (RESULTS_DIR / 'best_tau_surrogate.yaml').open('w', encoding='utf-8') as handle:
    yaml.safe_dump(payload, handle, sort_keys=False)
print(yaml.safe_dump(payload, sort_keys=False))

tau_nom_h:
  C1: 2.080770216460065
  C2: 30.108912662777
  C3: 19.243375568601316
  C4: 82.96307868162094
  C5: 23.52220990967777
  C6: 775.7036912161778
validated_on: test printers (id 85..99)
surrogate_annual_cost: 135043.85112705317
real_annual_cost_eur_per_printer_year: 4044316.6965507804
real_availability: 0.0

